# 14. HuggingFace Transformers

**Цель:** Познакомиться с библиотекой HuggingFace Transformers: загрузить предобученные BERT и GPT-2, выполнить fine-tuning для классификации, сравнить с нашей реализацией.

---

In [ ]:
import sys, os, logging, math
LOG_LEVEL = os.getenv("LOG_LEVEL", "DEBUG")
logging.basicConfig(level=getattr(logging, LOG_LEVEL), format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", stream=sys.stderr)
log = logging.getLogger("hf")

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
log.info("Using device: %s", device)

In [ ]:
log.info("Loading HuggingFace libraries")

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM,
    pipeline, Trainer, TrainingArguments, BertForSequenceClassification
)
from datasets import load_dataset

log.info("HuggingFace libraries loaded")

## 14.1 BERT: загрузка предобученной модели и токенизация

In [ ]:
log.info("Loading BERT tokenizer and model (distilbert for speed)")

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

log.info("BERT model loaded: %s, params=%d", model_name, sum(p.numel() for p in bert_model.parameters()))

# Тест токенизации
text = "Transformers are amazing for natural language processing!"
tokens = tokenizer(text, return_tensors="pt").to(device)
print(f"Original: {text}")
print(f"Tokens:   {tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])}")
print(f"Input IDs: {tokens['input_ids'][0].tolist()}")

## 14.2 Fine-tuning BERT на IMDb

Обучим BERT на задаче классификации тональности (sentiment analysis).

In [ ]:
log.info("Loading IMDb dataset (small subset)")

# Загружаем маленькую часть для демонстрации
dataset = load_dataset("imdb", split=["train[:100]", "test[:50]"])
train_dataset, test_dataset = dataset

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples:  {len(test_dataset)}")
print(f"Example: {train_dataset[0]['text'][:100]}...")
print(f"Label: {train_dataset[0]['label']} (0=neg, 1=pos)")
log.info("IMDb dataset loaded")

In [ ]:
log.debug("Tokenizing IMDb dataset")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

train_enc = train_dataset.map(tokenize_function, batched=True)
test_enc = test_dataset.map(tokenize_function, batched=True)

train_enc = train_enc.rename_column("label", "labels")
test_enc = test_enc.rename_column("label", "labels")
train_enc.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_enc.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

log.info("IMDb tokenized: %d train, %d test", len(train_enc), len(test_enc))

In [ ]:
log.info("Fine-tuning BERT on IMDb (2 epochs)")

training_args = TrainingArguments(
    output_dir="./checkpoints/hf-bert-imdb",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=5,
    evaluation_strategy="steps",
    eval_steps=10,
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_enc,
    eval_dataset=test_enc,
)

trainer.train()
log.info("BERT fine-tuning complete")

In [ ]:
log.debug("Evaluating fine-tuned BERT")

results = trainer.evaluate()
print(f"Evaluation loss: {results['eval_loss']:.4f}")

# Тест на новых примерах
test_texts = [
    "This movie was absolutely fantastic! I loved every minute.",
    "Terrible waste of time. The acting was horrible.",
    "It was okay, not great but not terrible either.",
]
for text in test_texts:
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    pred = outputs.logits.argmax(-1).item()
    sentiment = "positive" if pred == 1 else "negative"
    print(f"  '{text[:50]}...' -> {sentiment}")
log.info("BERT evaluation complete")

## 14.3 GPT-2: генерация текста с предобученной моделью

In [ ]:
log.info("Loading GPT-2 for text generation")

gpt2_name = "gpt2"
gpt2_tokenizer = AutoTokenizer.from_pretrained(gpt2_name)
gpt2_model = AutoModelForCausalLM.from_pretrained(gpt2_name).to(device)
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

log.info("GPT-2 loaded: params=%d", sum(p.numel() for p in gpt2_model.parameters()))

prompt = "The future of artificial intelligence is"
inputs = gpt2_tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = gpt2_model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        pad_token_id=gpt2_tokenizer.eos_token_id,
    )

generated = gpt2_tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"Generated: {generated}")
log.info("GPT-2 generation complete")

## 14.4 Pipeline API

HuggingFace Pipeline — высокоуровневый API для быстрого запуска моделей.

In [ ]:
log.debug("Using HuggingFace pipelines")

# Text classification pipeline
classifier = pipeline("text-classification", model=bert_model, tokenizer=tokenizer, device=0 if device.type == 'mps' else -1)

samples = [
    "I really enjoyed this film, great acting!",
    "This was boring and way too long.",
]
for text, result in zip(samples, classifier(samples)):
    print(f"  '{text[:40]}...' -> {result['label']} (score: {result['score']:.3f})")

# Text generation pipeline
generator = pipeline("text-generation", model=gpt2_model, tokenizer=gpt2_tokenizer, device=0 if device.type == 'mps' else -1)
result = generator("Transformers have revolutionized", max_new_tokens=30, num_return_sequences=1)
print(f"  Generated: {result[0]['generated_text']}")
log.info("Pipeline API demo complete")

## 14.5 Сравнение: наша реализация vs HuggingFace

| Аспект | Наша реализация | HuggingFace |
|--------|----------------|-------------|
| Размер | ~50K параметров | ~67M (BERT-base) |
| Данные | Синтетические | Wikipedia + BookCorpus |
| Время обучения | Минуты | Дни на TPU |
| Качество | Базовое | SOTA |
| Гибкость | Полный контроль | Высокоуровневый API |
| Скорость inference | Медленнее | Оптимизировано (kernel fusion) |

**Вывод:** Наша реализация помогла понять механизмы изнутри. HuggingFace — для production.

In [ ]:
print("=== HuggingFace Transformers complete ===")
print("Topics covered:")
print("  - Loading BERT (distilbert) with AutoTokenizer/AutoModel")
print("  - Fine-tuning BERT for text classification on IMDb")
print("  - Trainer API")
print("  - GPT-2 text generation (top-p, temperature)")
print("  - HuggingFace Pipeline API")
print("  - Comparison: our implementation vs HuggingFace")
log.info("HuggingFace notebook complete")